# Transformer Training on Colab

This notebook runs the transformer training workflow on Google Colab. The training code is kept in `src/models/train_transformer.py`, while this notebook is only used to set up the Colab environment, prepare the data, and launch training on GPU.

In [1]:
import os
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [10]:
%cd /content

!rm -rf Sentiment-Analysis-IMDB-reviews
!git clone https://github.com/mihneacoman/Sentiment-Analysis-IMDB-reviews

%cd Sentiment-Analysis-IMDB-reviews

/content
Cloning into 'Sentiment-Analysis-IMDB-reviews'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 68 (delta 20), reused 37 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 581.62 KiB | 12.37 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/Sentiment-Analysis-IMDB-reviews


In [12]:
!ls

!ls src/models

notebooks  README.md  requirements.txt	src
__pycache__  train_transformer.py


## Load the dataset

In [13]:
from pathlib import Path
import shutil

drive_processed_dir = Path("/content/drive/MyDrive/Updated Sentiment Analysis/data/processed")
repo_processed_dir = Path("data/processed")

repo_processed_dir.mkdir(parents=True, exist_ok=True)

for filename in ["train.csv", "valid.csv", "test.csv"]:
    src = drive_processed_dir / filename
    dst = repo_processed_dir / filename

    if not src.exists():
        raise FileNotFoundError(f"Missing file in Drive: {src}")

    shutil.copy(src, dst)

print("Copied files:")
for path in repo_processed_dir.iterdir():
    print(path)

Copied files:
data/processed/valid.csv
data/processed/test.csv
data/processed/train.csv


In [14]:
import pandas as pd

train_df = pd.read_csv("data/processed/train.csv")
valid_df = pd.read_csv("data/processed/valid.csv")
test_df = pd.read_csv("data/processed/test.csv")

print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)

print(train_df["label"].value_counts())

(8000, 2)
(2000, 2)
(2000, 2)
label
0    4000
1    4000
Name: count, dtype: int64


## Install dependencies

In [7]:
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.6 MB/s eta 0:00:00


In [15]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU: Tesla T4


## Training DistilBert

In [16]:
!python -m src.models.train_transformer \
  --model-name distilbert-base-uncased \
  --output-dir models/distilbert-sentiment \
  --max-length 256 \
  --epochs 3 \
  --learning-rate 2e-5 \
  --batch-size 16

Map: 100% 8000/8000 [00:08<00:00, 927.48 examples/s] 
Map: 100% 2000/2000 [00:01<00:00, 1229.24 examples/s]
Loading weights: 100% 100/100 [00:00<00:00, 6877.37it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
{'loss': '0.5828', 'grad_norm': '3.917', 'learning_rat

In [17]:
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer

MODEL_DIR = "models/distilbert-sentiment"
TEST_PATH = "data/processed/test.csv"
MAX_LENGTH = 256

test_df = pd.read_csv(TEST_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

test_dataset = Dataset.from_pandas(test_df)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

test_dataset = test_dataset.map(tokenize_batch, batched=True)

trainer = Trainer(
    model=model,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)

predictions_output = trainer.predict(test_dataset)
y_pred = np.argmax(predictions_output.predictions, axis=-1)
y_true = predictions_output.label_ids

print("Test accuracy:", accuracy_score(y_true, y_pred))
print()
print(classification_report(
    y_true,
    y_pred,
    target_names=["negative", "positive"]
))

print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Test accuracy: 0.898

              precision    recall  f1-score   support

    negative       0.93      0.86      0.89      1000
    positive       0.87      0.93      0.90      1000

    accuracy                           0.90      2000
   macro avg       0.90      0.90      0.90      2000
weighted avg       0.90      0.90      0.90      2000

Confusion matrix:
[[864 136]
 [ 68 932]]


In [18]:
!mkdir -p "/content/drive/MyDrive/Updated Sentiment Analysis/models"
!cp -r models/distilbert-sentiment "/content/drive/MyDrive/Updated Sentiment Analysis/models/"

## Training BERT

In [22]:
!git pull

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 2), reused 5 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 867 bytes | 867.00 KiB/s, done.
From https://github.com/mihneacoman/Sentiment-Analysis-IMDB-reviews
   35f0e84..8228151  main       -> origin/main
Updating 35f0e84..8228151
Fast-forward
 src/models/train_transformer.py | 50 +++++++++++++++++++++++++++--------------
 1 file changed, 33 insertions(+), 17 deletions(-)


In [23]:
!python -m src.models.train_transformer \
  --model-name bert-base-uncased \
  --output-dir models/bert-sentiment \
  --max-length 256 \
  --epochs 3 \
  --learning-rate 2e-5 \
  --batch-size 16

config.json: 100% 570/570 [00:00<00:00, 2.15MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 232kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 1.83MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 1.33MB/s]
Map: 100% 8000/8000 [00:06<00:00, 1190.79 examples/s]
Map: 100% 2000/2000 [00:01<00:00, 1217.63 examples/s]
model.safetensors: 100% 440M/440M [00:05<00:00, 74.8MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 4283.81it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.pred

In [24]:
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer

MODEL_DIR = "models/bert-sentiment"
TEST_PATH = "data/processed/test.csv"
MAX_LENGTH = 256

test_df = pd.read_csv(TEST_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

test_dataset = Dataset.from_pandas(test_df)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

test_dataset = test_dataset.map(tokenize_batch, batched=True)

trainer = Trainer(
    model=model,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)

predictions_output = trainer.predict(test_dataset)
y_pred = np.argmax(predictions_output.predictions, axis=-1)
y_true = predictions_output.label_ids

print("Test accuracy:", accuracy_score(y_true, y_pred))
print()
print(classification_report(
    y_true,
    y_pred,
    target_names=["negative", "positive"]
))

print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Test accuracy: 0.9115

              precision    recall  f1-score   support

    negative       0.93      0.89      0.91      1000
    positive       0.90      0.93      0.91      1000

    accuracy                           0.91      2000
   macro avg       0.91      0.91      0.91      2000
weighted avg       0.91      0.91      0.91      2000

Confusion matrix:
[[891 109]
 [ 68 932]]


In [28]:
!ls -lh models/bert-sentiment

total 419M
drwxr-xr-x 2 root root 4.0K Jun 14 15:32 checkpoint-1000
drwxr-xr-x 2 root root 4.0K Jun 14 15:39 checkpoint-1500
drwxr-xr-x 2 root root 4.0K Jun 14 15:24 checkpoint-500
-rw-r--r-- 1 root root  816 Jun 14 15:39 config.json
-rw------- 1 root root 418M Jun 14 15:39 model.safetensors
-rw-r--r-- 1 root root  351 Jun 14 15:39 tokenizer_config.json
-rw-r--r-- 1 root root 695K Jun 14 15:39 tokenizer.json
-rw-r--r-- 1 root root 5.1K Jun 14 15:39 training_args.bin


In [25]:
!mkdir -p "/content/drive/MyDrive/Updated Sentiment Analysis/models"
!cp -r models/bert-sentiment "/content/drive/MyDrive/Updated Sentiment Analysis/models/"

In [27]:
%cd /content/Sentiment-Analysis-IMDB-reviews

!rm -rf bert-sentiment-final
!mkdir -p bert-sentiment-final

!cp models/bert-sentiment/config.json bert-sentiment-final/
!cp models/bert-sentiment/model.safetensors bert-sentiment-final/
!cp models/bert-sentiment/tokenizer.json bert-sentiment-final/
!cp models/bert-sentiment/tokenizer_config.json bert-sentiment-final/
!cp models/bert-sentiment/vocab.txt bert-sentiment-final/
!cp models/bert-sentiment/special_tokens_map.json bert-sentiment-final/ 2>/dev/null || true
!cp models/bert-sentiment/training_args.bin bert-sentiment-final/ 2>/dev/null || true

!tar -czf bert-sentiment-final.tar.gz bert-sentiment-final

!mkdir -p "/content/drive/MyDrive/Updated Sentiment Analysis/model_archives"
!cp bert-sentiment-final.tar.gz "/content/drive/MyDrive/Updated Sentiment Analysis/model_archives/"

/content/Sentiment-Analysis-IMDB-reviews
cp: cannot stat 'models/bert-sentiment/vocab.txt': No such file or directory
